# Setup

In [1]:
import datasets
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

raw_dataset = datasets.load_dataset("sookiemonster/asrs-narratives")

utils.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [2]:
train_ds = raw_dataset['train']
valid_ds = raw_dataset['validation']
test_ds = raw_dataset['test']

labels = train_ds.features['label'].names

id_to_label = { idx : label for idx, label in enumerate(labels) }
label_to_id = { label : idx for idx, label in id_to_label.items() }

In [3]:
from functools import partial

def filter_labels(ds, to_remove:list):
  to_remove_set = set(to_remove)
  return ds.filter(lambda example : id_to_label[example['label']] not in to_remove_set)

filter_ambiguous = partial(filter_labels, to_remove=['ambiguous'])

filtered_train_ds = filter_ambiguous(train_ds)
filtered_valid_ds = filter_ambiguous(valid_ds)

Filter:   0%|          | 0/10360 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4441 [00:00<?, ? examples/s]

In [4]:
from datasets import ClassLabel

groupings = {
    'ac' : set(['aircraft']),
    'not-ac' : set(id_to_label.values()) - set(['aircraft'])
}

def _validate_groupings(groupings:dict[str, set]):
  mut_excl = [va.isdisjoint(vb) for ka, va in groupings.items() for kb, vb in groupings.items() if ka != kb]
  assert all(mut_excl)

  all_labels = set([label for val_set in groupings.values() for label in val_set])
  assert all_labels == set(id_to_label.values()), f"Missing: {set(id_to_label.values()) - all_labels}"


def group_labels(ds, groupings:dict[str, set]):
  _validate_groupings(groupings)
  group_names = list(groupings.keys())
  group_names.sort()

  fine_grained_label_to_group = {
      label : group_name for group_name, val_set in groupings.items() for label in val_set
  }

  res = ds.map(lambda ex: {"group" : fine_grained_label_to_group[ id_to_label[ex['label']] ]})

  new_features = res.features.copy()
  new_features["group"] = ClassLabel(names=group_names)

  res = res.cast(new_features)
  return res

grouped_ds_train = group_labels(filtered_train_ds, groupings)
grouped_ds_valid = group_labels(filtered_valid_ds, groupings)

Map:   0%|          | 0/9525 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9525 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4083 [00:00<?, ? examples/s]

# Fine Tuning

In [5]:
import torch
import sys

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}")
print(f"PyTorch Version: {torch.__version__.split('+')[0]}")
print(f"CUDA Version (Torch): {torch.version.cuda}")
print(f"CXX11 ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}")

Python Version: 3.12
PyTorch Version: 2.10.0
CUDA Version (Torch): 12.8
CXX11 ABI: True


In [6]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 10.4 MB/s eta 0:00:00


In [7]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00


In [8]:
import evaluate

accuracy = evaluate.load("accuracy")
precison = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precison.compute(predictions=predictions, references=labels, zero_division=np.nan)["precision"],
        "recall": recall.compute(predictions=predictions, references=labels)["recall"]
    }

In [9]:
from transformers import AutoTokenizer

MODEL_ID = "nomic-ai/modernbert-embed-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess(ds:datasets):
    PREFIX = "classification: "
    res = ds.map(lambda examples : {"text" : PREFIX + examples["text"]})
    res = res.map(lambda examples : tokenizer(
        examples["text"],
        padding=True,
        return_tensors="pt"
    ), batched=True)
    res = res.remove_columns(["acn", "label", 'text']).rename_column('group', 'label')
    return res

tokenized_train = preprocess(grouped_ds_train)
tokenized_eval = preprocess(grouped_ds_valid)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/9525 [00:00<?, ? examples/s]

Map:   0%|          | 0/9525 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

In [10]:
id_to_group = { i: group for i, group in enumerate(tokenized_train.features['label'].names) }
group_to_id = { group: i for i, group in id_to_group.items()}

In [11]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    id2label=id_to_group,
    label2id=group_to_id,
    dtype=torch.bfloat16,
    num_labels=2,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device=device)

model.safetensors:   0%|          | 0.00/596M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: nomic-ai/modernbert-embed-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 
head.norm.weight  | MISSING | 
head.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./nomic-modernbert-finetuned",
    bf16=True,

    learning_rate=4e-5,
    num_train_epochs=4,
    weight_decay=0.01,

    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id="sookiemonster/asrs-modernbert-aircraft-vs-rest",

    report_to="none",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Run Trainer

In [19]:
print(f"PAD ID: {tokenizer.pad_token_id}")
print(f"EOS ID: {tokenizer.eos_token_id}")
print(f"BOS ID: {tokenizer.bos_token_id}")

PAD ID: 50283
EOS ID: None
BOS ID: None


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall
1,0.319417,0.280278,0.896645,0.917371,0.920200
2,0.271021,0.280651,0.901298,0.932491,0.910563
3,0.236526,0.269130,0.902523,0.920046,0.927140
4,0.235657,0.268944,0.901298,0.919893,0.925212


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=596, training_loss=0.266077683275978, metrics={'train_runtime': 471.5156, 'train_samples_per_second': 80.803, 'train_steps_per_second': 1.264, 'total_flos': 1.1179327045057216e+17, 'train_loss': 0.266077683275978, 'epoch': 4.0})